In [1]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python314\python.exe -m pip install --upgrade pip setuptools wheel -q

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [2]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [3]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [5]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:.2f}'.format)

In [6]:
%%time
query = """SELECT
    -- Claves y metadatos básicos
    DISTINCT(a.num_documento) AS num_documento,
    CAST(a.mes_base AS INTEGER) AS mes_base,
    a.codunico,
    a.tipo_documento,
    a.tipo_cliente,
    a.subsegmento,
    
    -- Fechas
    TRY_CAST(a.fecha_vinculacion AS DATE) AS fecha_vinculacion,
    TRY_CAST(a.fecha_nacimiento AS DATE) AS fecha_nacimiento,
    TRY_CAST(a.fecha_mas_antiguo_lsb AS DATE) AS fecha_mas_antiguo_lsb,
    TRY_CAST(a.fecha_mas_reciente_lsb AS DATE) AS fecha_mas_reciente_lsb,
    TRY_CAST(a.fecha_ejecucion AS DATE) AS fecha_ejecucion,
    TRY_CAST(a.ultima_fecha_r1 AS DATE) AS ultima_fecha_r1,
    TRY_CAST(a.fecha_ultimo_kyc_hist AS TIMESTAMP) AS fecha_ultimo_kyc_hist,
    TRY_CAST(a.fecha_constitucion AS DATE) AS fecha_constitucion,
    TRY_CAST(a.fecha_ultimo_caso AS TIMESTAMP) AS fecha_ultimo_caso,
    TRY_CAST(a.fecha_ultima_alerta AS TIMESTAMP) AS fecha_ultima_alerta,
    TRY_CAST(a.fecha_ultimo_ros AS TIMESTAMP) AS fecha_ultimo_ros,

    -- Numéricos / decimales
    TRY_CAST(a.pasivo_soles AS DOUBLE) AS pasivo_soles,
    TRY_CAST(a.trx_monto_abonos_1m_total AS DOUBLE) AS trx_monto_abonos_1m_total,
    TRY_CAST(a.trx_monto_abonos_3m_total AS DOUBLE) AS trx_monto_abonos_3m_total,
    TRY_CAST(a.trx_monto_abonos_6m_total AS DOUBLE) AS trx_monto_abonos_6m_total,
    TRY_CAST(a.trx_monto_abonos_9m_total AS DOUBLE) AS trx_monto_abonos_9m_total,
    TRY_CAST(a.trx_monto_abonos_12m_total AS DOUBLE) AS trx_monto_abonos_12m_total,
    TRY_CAST(a.trx_monto_cargos_1m_total AS DOUBLE) AS trx_monto_cargos_1m_total,
    TRY_CAST(a.trx_monto_cargos_3m_total AS DOUBLE) AS trx_monto_cargos_3m_total,
    TRY_CAST(a.trx_monto_cargos_6m_total AS DOUBLE) AS trx_monto_cargos_6m_total,
    TRY_CAST(a.trx_monto_cargos_9m_total AS DOUBLE) AS trx_monto_cargos_9m_total,
    TRY_CAST(a.trx_monto_cargos_12m_total AS DOUBLE) AS trx_monto_cargos_12m_total,
    TRY_CAST(a.trx_monto_abonos_1m_efectivo AS DOUBLE) AS trx_monto_abonos_1m_efectivo,
    TRY_CAST(a.trx_monto_abonos_3m_efectivo AS DOUBLE) AS trx_monto_abonos_3m_efectivo,
    TRY_CAST(a.trx_monto_abonos_6m_efectivo AS DOUBLE) AS trx_monto_abonos_6m_efectivo,
    TRY_CAST(a.trx_monto_abonos_9m_efectivo AS DOUBLE) AS trx_monto_abonos_9m_efectivo,
    TRY_CAST(a.trx_monto_abonos_12m_efectivo AS DOUBLE) AS trx_monto_abonos_12m_efectivo,
    TRY_CAST(a.trx_monto_cargos_1m_efectivo AS DOUBLE) AS trx_monto_cargos_1m_efectivo,
    TRY_CAST(a.trx_monto_cargos_3m_efectivo AS DOUBLE) AS trx_monto_cargos_3m_efectivo,
    TRY_CAST(a.trx_monto_cargos_6m_efectivo AS DOUBLE) AS trx_monto_cargos_6m_efectivo,
    TRY_CAST(a.trx_monto_cargos_9m_efectivo AS DOUBLE) AS trx_monto_cargos_9m_efectivo,
    TRY_CAST(a.trx_monto_cargos_12m_efectivo AS DOUBLE) AS trx_monto_cargos_12m_efectivo,
    
    -- Cantidades de transacciones (int)
    TRY_CAST(a.trx_q_abonos_1m_total AS INTEGER) AS trx_q_abonos_1m_total,
    TRY_CAST(a.trx_q_abonos_3m_total AS INTEGER) AS trx_q_abonos_3m_total,
    TRY_CAST(a.trx_q_abonos_6m_total AS INTEGER) AS trx_q_abonos_6m_total,
    TRY_CAST(a.trx_q_abonos_9m_total AS INTEGER) AS trx_q_abonos_9m_total,
    TRY_CAST(a.trx_q_abonos_12m_total AS INTEGER) AS trx_q_abonos_12m_total,
    TRY_CAST(a.trx_q_cargos_1m_total AS INTEGER) AS trx_q_cargos_1m_total,
    TRY_CAST(a.trx_q_cargos_3m_total AS INTEGER) AS trx_q_cargos_3m_total,
    TRY_CAST(a.trx_q_cargos_6m_total AS INTEGER) AS trx_q_cargos_6m_total,
    TRY_CAST(a.trx_q_cargos_9m_total AS INTEGER) AS trx_q_cargos_9m_total,
    TRY_CAST(a.trx_q_cargos_12m_total AS INTEGER) AS trx_q_cargos_12m_total,
    TRY_CAST(a.trx_q_abonos_1m_efectivo AS INTEGER) AS trx_q_abonos_1m_efectivo,
    TRY_CAST(a.trx_q_abonos_3m_efectivo AS INTEGER) AS trx_q_abonos_3m_efectivo,
    TRY_CAST(a.trx_q_abonos_6m_efectivo AS INTEGER) AS trx_q_abonos_6m_efectivo,
    TRY_CAST(a.trx_q_abonos_9m_efectivo AS INTEGER) AS trx_q_abonos_9m_efectivo,
    TRY_CAST(a.trx_q_abonos_12m_efectivo AS INTEGER) AS trx_q_abonos_12m_efectivo,
    TRY_CAST(a.trx_q_cargos_1m_efectivo AS INTEGER) AS trx_q_cargos_1m_efectivo,
    TRY_CAST(a.trx_q_cargos_3m_efectivo AS INTEGER) AS trx_q_cargos_3m_efectivo,
    TRY_CAST(a.trx_q_cargos_6m_efectivo AS INTEGER) AS trx_q_cargos_6m_efectivo,
    TRY_CAST(a.trx_q_cargos_9m_efectivo AS INTEGER) AS trx_q_cargos_9m_efectivo,
    TRY_CAST(a.trx_q_cargos_12m_efectivo AS INTEGER) AS trx_q_cargos_12m_efectivo,

    -- Promedios y ratios (ya estaban casi todos)
    TRY_CAST(a.trx_q_abonos_promedio_3m_total AS DOUBLE) AS trx_q_abonos_promedio_3m_total,
    TRY_CAST(a.trx_q_abonos_promedio_6m_total AS DOUBLE) AS trx_q_abonos_promedio_6m_total,
    TRY_CAST(a.trx_q_abonos_promedio_9m_total AS DOUBLE) AS trx_q_abonos_promedio_9m_total,
    TRY_CAST(a.trx_q_abonos_promedio_12m_total AS DOUBLE) AS trx_q_abonos_promedio_12m_total,
    TRY_CAST(a.trx_monto_abonos_promedio_3m_total AS DOUBLE) AS trx_monto_abonos_promedio_3m_total,
    TRY_CAST(a.trx_monto_abonos_promedio_6m_total AS DOUBLE) AS trx_monto_abonos_promedio_6m_total,
    TRY_CAST(a.trx_monto_abonos_promedio_9m_total AS DOUBLE) AS trx_monto_abonos_promedio_9m_total,
    TRY_CAST(a.trx_monto_abonos_promedio_12m_total AS DOUBLE) AS trx_monto_abonos_promedio_12m_total,
    TRY_CAST(a.trx_monto_cargos_promedio_3m_total AS DOUBLE) AS trx_monto_cargos_promedio_3m_total,
    TRY_CAST(a.trx_monto_cargos_promedio_6m_total AS DOUBLE) AS trx_monto_cargos_promedio_6m_total,
    TRY_CAST(a.trx_monto_cargos_promedio_9m_total AS DOUBLE) AS trx_monto_cargos_promedio_9m_total,
    TRY_CAST(a.trx_monto_cargos_promedio_12m_total AS DOUBLE) AS trx_monto_cargos_promedio_12m_total,
    TRY_CAST(a.trx_q_cargos_promedio_3m_total AS DOUBLE) AS trx_q_cargos_promedio_3m_total,
    TRY_CAST(a.trx_q_cargos_promedio_6m_total AS DOUBLE) AS trx_q_cargos_promedio_6m_total,
    TRY_CAST(a.trx_q_cargos_promedio_9m_total AS DOUBLE) AS trx_q_cargos_promedio_9m_total,
    TRY_CAST(a.trx_q_cargos_promedio_12m_total AS DOUBLE) AS trx_q_cargos_promedio_12m_total,

    -- Ratios efectivo/total
    TRY_CAST(a.trx_q_abonos_ratio_1m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_1m_efectivo_total,
    TRY_CAST(a.trx_q_abonos_ratio_3m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_3m_efectivo_total,
    TRY_CAST(a.trx_q_abonos_ratio_6m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_6m_efectivo_total,
    TRY_CAST(a.trx_q_abonos_ratio_9m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_9m_efectivo_total,
    TRY_CAST(a.trx_q_abonos_ratio_12m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_12m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_1m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_1m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_3m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_3m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_6m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_6m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_9m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_9m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_12m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_12m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_1m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_1m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_3m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_3m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_6m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_6m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_9m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_9m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_12m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_12m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_1m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_1m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_3m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_3m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_6m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_6m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_9m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_9m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_12m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_12m_efectivo_total,

    -- Máximos, diferencias, RO, exterior, etc.
    TRY_CAST(a.dif_monto_abonos_cargos_efectivo_6m AS DOUBLE) AS dif_monto_abonos_cargos_efectivo_6m,
    a.dif_q_abonos_cargos_efectivo_6m,
    TRY_CAST(a.monto_ro_debajo_umbral AS DOUBLE) AS monto_ro_debajo_umbral,
    TRY_CAST(a.trx_monto_abonos_1m_max AS DOUBLE) AS trx_monto_abonos_1m_max,
    TRY_CAST(a.trx_monto_abonos_3m_max AS DOUBLE) AS trx_monto_abonos_3m_max,
    TRY_CAST(a.trx_monto_abonos_6m_max AS DOUBLE) AS trx_monto_abonos_6m_max,
    TRY_CAST(a.trx_monto_abonos_9m_max AS DOUBLE) AS trx_monto_abonos_9m_max,
    TRY_CAST(a.trx_monto_abonos_12m_max AS DOUBLE) AS trx_monto_abonos_12m_max,
    TRY_CAST(a.trx_monto_cargos_1m_max AS DOUBLE) AS trx_monto_cargos_1m_max,
    TRY_CAST(a.trx_monto_cargos_3m_max AS DOUBLE) AS trx_monto_cargos_3m_max,
    TRY_CAST(a.trx_monto_cargos_6m_max AS DOUBLE) AS trx_monto_cargos_6m_max,
    TRY_CAST(a.trx_monto_cargos_9m_max AS DOUBLE) AS trx_monto_cargos_9m_max,
    TRY_CAST(a.trx_monto_cargos_12m_max AS DOUBLE) AS trx_monto_cargos_12m_max,

    TRY_CAST(a.monto_al_exterior_3m AS DOUBLE) AS monto_al_exterior_3m,
    TRY_CAST(a.monto_al_exterior_6m AS DOUBLE) AS monto_al_exterior_6m,
    TRY_CAST(a.monto_al_exterior_9m AS DOUBLE) AS monto_al_exterior_9m,
    TRY_CAST(a.monto_al_exterior_12m AS DOUBLE) AS monto_al_exterior_12m,
    TRY_CAST(a.monto_del_exterior_3m AS DOUBLE) AS monto_del_exterior_3m,
    TRY_CAST(a.monto_del_exterior_6m AS DOUBLE) AS monto_del_exterior_6m,
    TRY_CAST(a.monto_del_exterior_9m AS DOUBLE) AS monto_del_exterior_9m,
    TRY_CAST(a.monto_del_exterior_12m AS DOUBLE) AS monto_del_exterior_12m,

    -- Variables que ya venían limpias
    a.edad,
    a.edad_constitucion,
    a.antiguedad,
    a.cantidad_lsb,
    a.nivel_riesgo_lsb_total,
    a.nivel_riesgo_lsb_ultima,
    a.q_meses_ingresos_0,
    a.q_meses_egresos_0,
    a.provincia,
    a.departamento,
    a.ubigeo_cd,
    a.sectorista_id,
    a.ciiu_v4,

    -- Flags booleanos → 0/1
    CASE WHEN a.flag_casos_hist THEN 1 ELSE 0 END AS flag_casos_hist,
    CASE WHEN a.flag_desv_activa THEN 1 ELSE 0 END AS flag_desv_activa,
    CASE WHEN a.flag_ros_hist THEN 1 ELSE 0 END AS flag_ros_hist,
    CASE WHEN a.flag_variacion_abono_monto_total_5m_1m THEN 1 ELSE 0 END AS flag_variacion_abono_monto_total_5m_1m,
    CASE WHEN a.flag_variacion_efect_cargos_monto_5m_1m THEN 1 ELSE 0 END AS flag_variacion_efect_cargos_monto_5m_1m,

    -- Flags que ya son int
    a.flag_cce_r1,
    a.flag_kyc_12m,
    a.flag_kyc_hist,
    a.cantidad_kyc_hist,
    a.cp_cantidad_ing,
    a.q_ro_debajo_umbral,
    a.q_trx_al_exterior_3m, a.q_trx_al_exterior_6m, a.q_trx_al_exterior_9m, a.q_trx_al_exterior_12m,
    a.q_trx_del_exterior_3m, a.q_trx_del_exterior_6m, a.q_trx_del_exterior_9m, a.q_trx_del_exterior_12m,
    a.q_meses_al_exterior_0, a.q_meses_al_exterior_100, a.q_meses_al_exterior_1000,
    a.q_meses_al_exterior_10000, a.q_meses_al_exterior_100000, a.q_meses_al_exterior_1000000,
    a.flag_al_exterior,
    a.flag_del_exterior,
    a.flag_inteligo,

    -- NUEVAS COLUMNAS que faltaban
    a.nacionalidad,
    a.tipo_cliente_sensible,
    TRY_CAST(a.score_con_excepcion AS DOUBLE) AS score_con_excepcion,
    a.tipobancadesc,
    a.flag_casos_12m,
    a.estado_ultimo_caso,
    a.q_casos_hist,
    a.flag_desv_hist,
    a.estado_ultima_desv,
    a.periodo_inicio_desv_max,
    a.periodo_cierre_desv_max,
    a.meses_en_proceso_desvinculacion,
    a.flag_alerta_12m,
    a.flag_alerta_hist,
    a.calificacion_ultima_alerta,
    a.q_alerta_hist,
    a.flag_ros_12m,
    a.q_ros_hist,
    TRY_CAST(a.monto_trx_debajo_10k_ing AS DOUBLE) AS monto_trx_debajo_10k_ing,
    a.cant_trx_debajo_10k_ing,
    a.cant_trx_debajo_10k_egr,
    TRY_CAST(a.monto_trx_debajo_10k_egr AS DOUBLE) AS monto_trx_debajo_10k_egr,
    TRY_CAST(a.cp_cant_trx_egr_desv AS DOUBLE) AS cp_cant_trx_egr_desv,
    TRY_CAST(a.cp_monto_total_egr_desv AS DOUBLE) AS cp_monto_total_egr_desv,
    TRY_CAST(a.cp_cant_trx_ing_ros AS DOUBLE) AS cp_cant_trx_ing_ros,
    TRY_CAST(a.cp_monto_total_ing_ros AS DOUBLE) AS cp_monto_total_ing_ros,
    TRY_CAST(a.cp_cant_trx_egr_ros AS DOUBLE) AS cp_cant_trx_egr_ros,
    TRY_CAST(a.cp_monto_total_egr_ros AS DOUBLE) AS cp_monto_total_egr_ros,

    a.p_codmes,
    b.tipo_alerta_n2,

    -- TARGET
    CASE WHEN a.flg_alerta = '1' THEN 1 ELSE 0 END AS target_m


FROM d_perm_aws.ds_alertplaft_mdl a
LEFT JOIN e_perm_aws.t_alertas_plaft b
  ON a.codunico = b.codunico
 AND a.mes_base = b.periodo_alerta
WHERE a.mes_base BETWEEN '202501' AND '202509'
  AND a.subsegmento = 'BPE'

;"""
df = athena_query(query, database='disc_comercial')
df.head()

CPU times: total: 43.8 s
Wall time: 6min 22s


,num_documento,mes_base,codunico,tipo_documento,tipo_cliente,subsegmento,fecha_vinculacion,fecha_nacimiento,fecha_mas_antiguo_lsb,fecha_mas_reciente_lsb,...,monto_trx_debajo_10k_egr,cp_cant_trx_egr_desv,cp_monto_total_egr_desv,cp_cant_trx_ing_ros,cp_monto_total_ing_ros,cp_cant_trx_egr_ros,cp_monto_total_egr_ros,p_codmes,tipo_alerta_n2,target_m
0,96751C5173AE3E6C62A0EBFA3EDBA45CF04E959F5F268A...,202507,0019846324,2,ORGANIZATION,BPE,2023-04-13,None,2024-04-18,2024-04-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,202507,<NA>,0
1,EE1977AD0B55549B17D68B670AFD5E030975AA81F6C953...,202509,0012968605,2,ORGANIZATION,BPE,2012-09-07,None,2024-04-18,2024-04-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,202509,<NA>,0
2,B2109EFF2B14C37ABF5F5D0F7AD6011FBF43E457AF0ACD...,202506,0002430908,2,ORGANIZATION,BPE,1994-08-25,None,2024-04-18,2024-04-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,202506,<NA>,0
3,6896178B266744AE3A628353979B50E6505456D5C47719...,202504,0021143628,2,ORGANIZATION,BPE,2024-09-03,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,202504,<NA>,0
4,AB0B3CEBA566A3A3AB63EEC06A22ED33BD3F2BB46FFC7A...,202507,0015477679,2,ORGANIZATION,BPE,2018-01-17,None,2024-04-18,2024-04-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,202507,<NA>,0


In [7]:
df.shape

(1433597, 187)

In [8]:
df.nivel_riesgo_lsb_total.value_counts()

nivel_riesgo_lsb_total
ALTO    816097
Name: count, dtype: Int64

In [9]:
df['nivel_riesgo_lsb_total'] = df['nivel_riesgo_lsb_total'].map({
    'ALTO':1,
    'MEDIO':2,
    'BAJO':3,
    }).fillna(-1)

In [10]:
df['nivel_riesgo_lsb_ultima'] = df['nivel_riesgo_lsb_ultima'].map({
    'ALTO':1,
    'MEDIO':2,
    'BAJO':3,
    }).fillna(-1)

In [11]:
df.flag_casos_hist.value_counts()

flag_casos_hist
0    1432781
1        816
Name: count, dtype: Int64

In [12]:
df["cantidad_lsb"] =df["cantidad_lsb"].fillna(0)

In [13]:
# df_dataset: datos correspondientes a 202405
# df_dataset = pd.read_csv('C:/Users/X15403/Desktop/PLAFT/exploratorio/t_dataset_plaft_202405.csv', encoding='utf-8', sep=',', low_memory=False)

#df_dataset = pd.read_csv('C:/Users/X15403/Desktop/PLAFT/exploratorio/preprocesamiento.csv', encoding='utf-8', sep=',', low_memory=False)
#df_dataset = df_dataset.rename(columns={'flg_alerta': 'target_m'})
#df_dataset = df_dataset.drop_duplicates()

In [14]:
df.tipo_alerta_n2.value_counts()

tipo_alerta_n2
AUTOMATICA         4496
MANUAL              793
SEMI AUTOMATICA     548
Name: count, dtype: Int64

# Definición de funciones

In [15]:
from typing import List, Tuple

# Funcion columnas nulas
########################

def filtro_columnas_nulas(df: pd.DataFrame,
                          umbral_nulos: float,
                          col_conservar: List[str] = None,
                          col_eliminar: List[str] = None,
                          col_target: str = 'target_m'
                         ) -> pd.DataFrame:
    """
    Elimina columnas con más del porcentaje de nulos especificado, columnas con un único valor, y filas duplicadas.
    Las columnas en 'col_conservar' y la columna objetivo 'col_target' nunca se eliminan si existen en el DataFrame.

    Parámetros:
    - df: DataFrame a limpiar.
    - umbral_nulos: Porcentaje máximo permitido de nulos por columna (ej. 0.2 para 20%).
    - col_conservar: Lista de nombres de columnas que deben conservarse si existen.
    - col_conservar: Lista de nombres de columnas a eliminar.
    - col_target: Nombre de la columna objetivo que debe conservarse.

    Retorna:
    - DataFrame limpio.
    """
    print("\n\033[1m\033[4mFILTRANDO COLUMNAS...\033[0m\n")
    print(f"Shape inicial del df: {df.shape}\n")

    # Paso 0: Eliminar columnas especificadas en col_eliminar si existen
    if col_eliminar:
        cols_eliminar_existentes = [col for col in col_eliminar if col in df.columns]
        if cols_eliminar_existentes:
            df = df.drop(columns=cols_eliminar_existentes)
            print("Se eliminan del df las siguientes columnas:")
            for col in cols_eliminar_existentes:
                print(f" - {col}")
            print("")
        else:
            print("Ninguna de las columnas especificadas en col_eliminar existe en el DataFrame.\n")
    
    # Validación del umbral
    if not 0 <= umbral_nulos <= 1:
        raise ValueError("El umbral de nulos debe estar entre 0 y 1.")

    # Inicializar lista de columnas a conservar
    col_conservar = col_conservar or []

    # Asegurar que col_target esté en la lista de columnas a conservar
    if col_target not in col_conservar:
        col_conservar.append(col_target)

    # Filtrar solo las columnas que existen en el DataFrame
    col_conservar_existentes = [col for col in col_conservar if col in df.columns]

    # Mensaje de columnas que se conservarán
    print("Listado de columnas a conservar (no se aplica filtro segun porcentaje de datos faltates):")
    for col in col_conservar_existentes:
        print(f" - {col}")

    # Eliminar columnas con un único valor, preservando las columnas a conservar
    columnas_valor_unico = [
        col for col in df.columns
        if df[col].nunique() == 1 and col not in col_conservar_existentes
    ]
    print("")
    print("Se eliminan las siguientes columnas por tener valor único:")
    for col in columnas_valor_unico:
        print(f" - {col}")
        
    df = df.drop(columns=columnas_valor_unico)

    # Calcular porcentaje de nulos por columna
    porcentaje_nulos = df.isnull().mean()

    # Filtrar columnas que cumplen el umbral, preservando las columnas a conservar
    columnas_validas = [
        col for col in df.columns
        if (porcentaje_nulos[col] < umbral_nulos or col in col_conservar_existentes)
    ]
    df_filtrado = df[columnas_validas]
    print(f"\nSe eliminan {str(len(df.columns) - len(df_filtrado.columns))} columnas con más del {umbral_nulos*100}% de nulos")

    # Eliminar filas duplicadas
    df_filtrado = df_filtrado.drop_duplicates()
    
    print(f"\nShape del df: {df_filtrado.shape}")

    return df_filtrado

In [16]:
# Funcion target
################

def generar_target(df: pd.DataFrame, escenario: int, porcentaje_sampleo: float = 0.1) -> pd.DataFrame:
    """
    Genera o modifica la columna 'target_m' en el DataFrame según el escenario elegido.

    Escenarios:
    1 - Elimina registros con target nulo y convierte a entero.
    2 - Mantiene alertas con riesgo y agrega muestra de casos sin alertas (target nulo).
    3 - Solo mantiene alertas con riesgo y muestra de casos sin alertas.

    Parámetros:
    - df: DataFrame original.
    - escenario: Número de escenario (1, 2 o 3).
    - porcentaje_sampleo: Porcentaje de registros nulos a muestrear (valor entre 0 y 1).

    Retorna:
    - DataFrame modificado con columna 'target_m' procesada.
    """
    print("\n\033[1m\033[4mCREANDO TARGET...\033[0m\n")

    print("Escenarios posibles de creacion de target: ")
    print("1 - Elimina registros con target nulo y convierte a entero.")
    print("2 - Mantiene alertas con riesgo y agrega muestra de casos sin alertas (target nulo).")
    print("3 - Solo mantiene alertas con riesgo y muestra de casos sin alertas.")

    print(f"\nEscenario elegido : {escenario}")
    
    if escenario not in [1, 2, 3]:
        raise ValueError("El escenario debe ser 1, 2 o 3.")
    if not 0 < porcentaje_sampleo <= 1:
        raise ValueError("El porcentaje de sampleo debe estar entre 0 y 1.")

    df = df.copy()

    if escenario == 1:
        # Eliminar nulos y convertir a entero
        df = df[~df['target_m'].isnull()]
        df['target_m'] = df['target_m'].astype(int)

    else:
        # Escenarios 2 y 3: muestreo de registros con target nulo
        columns_to_check = [col for col in df.columns if col != 'target_m']
        sample_df = df[df['target_m'].isnull()].copy()
        sample_df['null_count'] = sample_df[columns_to_check].isnull().sum(axis=1)
        sorted_sample = sample_df.sort_values(by='null_count')

        n = int(len(sample_df) * porcentaje_sampleo)
        df_null = sorted_sample.head(n)

        if escenario == 2:
            df = pd.concat([df[~df['target_m'].isnull()], df_null])
        elif escenario == 3:
            df_target_1 = df[df['target_m'] == 1]
            df = pd.concat([df_target_1, df_null])

        df['target_m'] = df['target_m'].fillna(0)
        df['target_m'] = df['target_m'].astype(int)

    df = df.drop(columns=['null_count'])

    print(f"\nShape del df: {df.shape}")
    print("\nDistribución de la target:")
    print(df['target_m'].value_counts())

    return df



In [17]:
# Funcion para clasificar columnas
##################################

def clasificar_columnas(df: pd.DataFrame, 
                        target_variable: str):
    """
    Clasifica las columnas de un DataFrame en listas de variables target, fechas, categóricas y numéricas.
    Elimina previamente columnas que tengan valor unico.
    Elimina filas duplicadas.

    Parámetros:
    - df (pd.DataFrame): DataFrame de entrada.
    - target_variable (str): Nombre de la variable objetivo.
    - col_eliminar (list, opcional): Lista de columnas a eliminar explícitamente.

    Retorna:
    tuple: col_target, col_fechas, col_categoricas, col_numericas, df (modificado)
    """
    print("\n\033[1m\033[4mCLASIFICANDO COLUMNAS...\033[0m\n")

    # Paso 1: Eliminar columnas que tienen un único valor
    cols_to_drop = [col for col in df.columns if df[col].nunique() == 1]
    print("Se eliminan las siguientes columnas por tener valor único:")
    for col in cols_to_drop:
        print(f" - {col}")

    if 'key_value' in df.columns:
        cols_to_drop.append('key_value')
    df = df.drop(columns=cols_to_drop)

    print("")
    print(f"Shape después de eliminar columnas: {df.shape}")

    # Paso 2: Eliminar filas duplicadas
    df = df.drop_duplicates()
    print(f"Shape después de eliminar filas duplicadas: {df.shape}")
    print("")

    # Paso 3: Identificar la variable target
    col_target = [target_variable] if target_variable in df.columns else []

    # Paso 4: Identificar variables categóricas y numéricas
    col_categoricas = df.select_dtypes(include='object').columns.tolist()
    col_categoricas = [col for col in col_categoricas if col not in col_target]

    col_numericas = df.select_dtypes(include=['int64', 'float64','int32','number']).columns.tolist()
    col_numericas = [col for col in col_numericas if col not in col_target]

    # Paso 5: Corrección de clasificación usando substrings
    substrings_categoricas = ['flag', 'lugar', 'tipo', 'cod', 'ciiu_v4', 'ubigeo_cd']
    moved_to_categoricas = [col for col in col_numericas if any(sub in col.lower() for sub in substrings_categoricas)]
    col_categoricas.extend(moved_to_categoricas)
    col_numericas = [col for col in col_numericas if col not in moved_to_categoricas]

    # Paso 5.2: Corrección de clasificación usando substrings
    substrings_numericas = ['v13_lugar_operativa_riesgo', 'v16_canal_operacion_riesgo']
    moved_to_numericas = [col for col in col_categoricas if any(sub in col.lower() for sub in substrings_numericas)]
    col_numericas.extend(moved_to_numericas)
    col_categoricas = [col for col in col_categoricas if col not in moved_to_numericas]

    # Paso 6: Identificación de variables de fecha
    col_fechas = []
    substrings_fecha = ['fecha', 'mes_base']
    moved_from_categoricas = [col for col in col_categoricas if any(sub in col.lower() for sub in substrings_fecha)]
    col_fechas.extend(moved_from_categoricas)
    col_categoricas = [col for col in col_categoricas if col not in col_fechas]

    moved_from_numericas = [col for col in col_numericas if any(sub in col.lower() for sub in substrings_fecha)]
    col_fechas.extend(moved_from_numericas)
    col_numericas = [col for col in col_numericas if col not in col_fechas]

    # Paso 7: Reemplazar '0.0' por '0' y '1.0' por '1' en columnas categóricas de flag 
    for col in col_categoricas:
        df[col] = df[col].astype('str')
        if col in df.columns and 'flag_' in col:
            df[col] = df[col].replace({'0.0': '0', '1.0': '1'})

    # Impresión de listas clasificadas con cantidad de elementos
    print(f"Variables de fecha ({len(col_fechas)}):")
    for col in col_fechas:
        print(f" - {col}")
    print()

    print(f"Variables categóricas ({len(col_categoricas)}):")
    for col in col_categoricas:
        print(f" - {col}")
    print()

    print(f"Variables numéricas ({len(col_numericas)}):")
    for col in col_numericas:
        print(f" - {col}")
    print()

    # Retornar las listas y el DataFrame modificado
    return col_target, col_fechas, col_categoricas, col_numericas, df

In [18]:
# Funcion para imputar nulos en columnas categoricas
####################################################

def imputar_categoricas(df: pd.DataFrame,
                        col_categoricas: List[str],
                        criterio_riesgo: str = 'sin_dato',
                        criterio_flag: str = 'sin_dato',
                        criterio_cat: str = 'sin_dato'
                       ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Imputa valores faltantes en columnas categóricas según criterios definidos.

    Parámetros:
    - df: DataFrame original.
    - col_categoricas: Lista de nombres de columnas categóricas.
    - criterio_riesgo: Valor para imputar columnas que contienen 'nivel_riesgo'.
    - criterio_flag: Valor para imputar columnas que contienen 'flag_'.
    - criterio_cat: Valor para imputar el resto ('sin_dato' o 'moda').

    Retorna:
    - df_imputado: DataFrame con imputaciones realizadas.
    - df_resumen: DataFrame resumen con criterios y cantidad de imputaciones por columna.
    """
    print("\n\033[1m\033[4mIMPUTANDO NULOS PARA COLUMNAS CATEGORICAS...\033[0m\n")
    df_imputado = df.copy()
    resumen = []

    for col in col_categoricas:
        if col not in df_imputado.columns:
            continue  # Saltear columnas que no existen en el DataFrame

        # Convertir a string para asegurar tipo
        df_imputado[col] = df_imputado[col].astype(str)

        nulos_antes = df_imputado[col].isna().sum() + (df_imputado[col] == 'nan').sum()

        if 'nivel_riesgo' in col:
            valor_imputacion = criterio_riesgo
            df_imputado[col] = df_imputado[col].replace('nan', pd.NA)
            df_imputado[col] = df_imputado[col].fillna(valor_imputacion)

        elif 'flag_' in col:
            valor_imputacion = criterio_flag
            df_imputado[col] = df_imputado[col].replace('nan', pd.NA)
            df_imputado[col] = df_imputado[col].fillna(valor_imputacion)

        else:
            if criterio_cat == 'sin_dato':
                valor_imputacion = 'sin_dato'
            elif criterio_cat == 'moda':
                moda = df_imputado[col].mode(dropna=True)
                valor_imputacion = moda[0] if not moda.empty else 'sin_dato'
            else:
                valor_imputacion = 'sin_dato'  # fallback

            df_imputado[col] = df_imputado[col].replace('nan', pd.NA)
            df_imputado[col] = df_imputado[col].fillna(valor_imputacion)

        # Asegurar tipo string después de imputar
        df_imputado[col] = df_imputado[col].astype(str)

        nulos_despues = df_imputado[col].isna().sum() + (df_imputado[col] == 'nan').sum()
        imputados = nulos_antes - nulos_despues

        resumen.append({
            'columna': col,
            'criterio_usado': valor_imputacion,
            'valores_imputados': imputados
        })

    df_resumen = pd.DataFrame(resumen)

    print('Resumen\n')
    display(df_resumen)

    return df_imputado, df_resumen


In [19]:
# Funcion para imputar nulos en columnas numércias
##################################################

def imputar_numericas(df: pd.DataFrame,
                      col_numericas: List[str],
                      criterio_dif: str,
                      criterio_trx: str,
                      criterio_q: str,
                      criterio_num: str
                     ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Imputa valores nulos en columnas numéricas según criterios definidos por tipo de variable.

    Parámetros:
    ----------
    df : pd.DataFrame
        DataFrame original.
    col_numericas : List[str]
        Lista de nombres de columnas numéricas a imputar.
    criterio_dif : str
        Criterio para columnas que contienen 'dif_'. Opciones: '0', 'media', 'max', 'p50'.
    criterio_trx : str
        Criterio para columnas que contienen 'trx_'. Opciones: '0', 'media', 'max', 'p50'.
    criterio_q : str
        Criterio para columnas que contienen 'q_'. Opciones: '0', 'media', 'max', 'p50'.
    criterio_num : str
        Criterio para el resto de las columnas. Opciones: '0', 'media', 'max', 'p50'.

    Retorna:
    -------
    df_imputado : pd.DataFrame
        DataFrame con valores imputados.
    df_resumen : pd.DataFrame
        DataFrame resumen con variable, criterio usado y cantidad de valores imputados.
    """
    print("\n\033[1m\033[4mIMPUTANDO NULOS PARA COLUMNAS NUMERICAS...\033[0m\n")
    df_imputado = df.copy()
    resumen = []

    criterios_validos = {"0", "media", "max", "p50"}

    def obtener_valor_imputacion(col: pd.Series, criterio: str):
        if criterio not in criterios_validos:
            raise ValueError(f"Criterio inválido: {criterio}")
        if criterio == "0":
            return 0
        elif criterio == "media":
            return col.mean()
        elif criterio == "max":
            return col.max()
        elif criterio == "p50":
            return col.quantile(0.5)

    for col in col_numericas:
        if col not in df_imputado.columns:
            continue  # O podrías loggear una advertencia
        if not pd.api.types.is_numeric_dtype(df_imputado[col]):
            continue

        nulos_antes = df_imputado[col].isna().sum()
        if nulos_antes == 0:
            continue

        if "dif_" in col:
            criterio = criterio_dif
        elif "trx_" in col:
            criterio = criterio_trx
        elif "q_" in col:
            criterio = criterio_q
        else:
            criterio = criterio_num

        valor_imputacion = obtener_valor_imputacion(df_imputado[col], criterio)
        df_imputado[col] = df_imputado[col].fillna(valor_imputacion)

        resumen.append({
            "variable": col,
            "criterio_usado": criterio,
            "valores_imputados": nulos_antes
        })

    df_resumen = pd.DataFrame(resumen)
    print('Resumen\n')
    display(df_resumen)
    return df_imputado, df_resumen

In [20]:
# Funcion para reagrupar variables categoricas
##############################################

def reagrupar_categorias(df, col_categoricas, target_m):
    """
    Reagrupa niveles de variables categóricas segun cantidad de casos positivos y efectividad.

    Parámetros:
    ----------
    df : pd.DataFrame
        DataFrame original.
    col_categoricas : List[str]
        Lista de nombres de columnas categóricas a reagrupar.
    target_m : str
        Variable target.

    Retorna:
    -------
    df_resultado : pd.DataFrame
        DataFrame con valores reagrupados.
    df_resumen : pd.DataFrame
        DataFrame resumen con variable, nuevo grupo, categoría original, detalle.
    diccionario:
        variabe: nuevo grupo: categoria original
    """
    print("\n\033[1m\033[4mREAGRUPANDO VARIABLES CATEGORICAS...\033[0m\n")
    
    df_resultado = df.copy()
    resumen = []

    # eliminamos tipo de alerta de las columnas categoricas
    if 'tipo_alerta_n2' in col_categoricas:
        col_categoricas.remove('tipo_alerta_n2')

    for col in col_categoricas:
        categorias = df[col].dropna().unique()
        if len(categorias) <= 3:
            continue
        # Calcular casos positivos y negativos por categoría
        stats = df.groupby(col)[target_m].agg(['sum', 'count'])
        stats['casos_positivos'] = stats['sum']
        stats['casos_negativos'] = stats['count'] - stats['sum']
        stats['efectividad'] = stats.apply(
            lambda row: row['casos_positivos'] / row['casos_negativos'] if row['casos_negativos'] > 0 else float('inf'),
            axis=1
        )
        stats = stats[['casos_positivos', 'casos_negativos', 'efectividad']]
        stats = stats.reset_index()

        # Escenario 1: todas las categorías tienen casos_positivos = 0
        if (stats['casos_positivos'] == 0).all():
            mapping = {cat: f"{col}_nulo" for cat in categorias}
            resumen.append({
                'variable': col,
                'grupo': f"{col}_nulo",
                'categorias_originales': list(categorias),
                'detalle': 'Todas las categorías con casos_positivos = 0'
            })

        # Escenario 2: algunas categorías con casos_positivos = 0, otras > 0
        elif (stats['casos_positivos'] == 0).any():
            nulo_cats = stats[stats['casos_positivos'] == 0][col].tolist()
            positivas = stats[stats['casos_positivos'] > 0].sort_values(by='efectividad', ascending=False)
            pos_cats = positivas[col].tolist()

            mapping = {cat: f"{col}_nulo" for cat in nulo_cats}

            if len(pos_cats) == 1:
                mapping[pos_cats[0]] = f"{col}_medio"
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_medio",
                    'categorias_originales': [pos_cats[0]],
                    'detalle': 'Una categoría con casos_positivos > 0'
                })
            else:
                mitad = len(pos_cats) // 2
                alto = pos_cats[:mitad]
                medio = pos_cats[mitad:]
                for cat in alto:
                    mapping[cat] = f"{col}_alto"
                for cat in medio:
                    mapping[cat] = f"{col}_medio"
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_alto",
                    'categorias_originales': alto,
                    'detalle': 'Categorías con alta efectividad'
                })
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_medio",
                    'categorias_originales': medio,
                    'detalle': 'Categorías con efectividad media'
                })
            resumen.append({
                'variable': col,
                'grupo': f"{col}_nulo",
                'categorias_originales': nulo_cats,
                'detalle': 'Categorías con casos_positivos = 0'
            })

        # Escenario 3: todas las categorías tienen casos_positivos >= 1
        else:
            stats_sorted = stats.sort_values(by='efectividad', ascending=False)
            sorted_cats = stats_sorted[col].tolist()
            n = len(sorted_cats)

            if n == 1:
                mapping = {sorted_cats[0]: f"{col}_medio"}
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_medio",
                    'categorias_originales': [sorted_cats[0]],
                    'detalle': 'Una sola categoría con casos_positivos >= 1'
                })
            elif n == 2:
                mapping = {
                    sorted_cats[0]: f"{col}_alto",
                    sorted_cats[1]: f"{col}_bajo"
                }
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_alto",
                    'categorias_originales': [sorted_cats[0]],
                    'detalle': 'Categoría con mayor efectividad'
                })
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_bajo",
                    'categorias_originales': [sorted_cats[1]],
                    'detalle': 'Categoría con menor efectividad'
                })
            else:
                tercio = n // 3
                alto = sorted_cats[:tercio]
                medio = sorted_cats[tercio:2*tercio]
                bajo = sorted_cats[2*tercio:]
                mapping = {}
                for cat in alto:
                    mapping[cat] = f"{col}_alto"
                for cat in medio:
                    mapping[cat] = f"{col}_medio"
                for cat in bajo:
                    mapping[cat] = f"{col}_bajo"
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_alto",
                    'categorias_originales': alto,
                    'detalle': 'Tercio superior de efectividad'
                })
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_medio",
                    'categorias_originales': medio,
                    'detalle': 'Tercio medio de efectividad'
                })
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_bajo",
                    'categorias_originales': bajo,
                    'detalle': 'Tercio inferior de efectividad'
                })

        # Aplicar el mapeo al DataFrame
        df_resultado[col] = df[col].map(mapping).fillna(df[col])

    df_resumen = pd.DataFrame(resumen)

    print('Resumen\n')
    display(df_resumen)

    # diccionario con las categorias reagrupadas
    diccionario = {}
    
    for _, row in df_resumen.iterrows():
        variable = row['variable']
        grupo = row['grupo']
        categorias = row['categorias_originales']
        
        if variable not in diccionario:
            diccionario[variable] = {}
        
        diccionario[variable][grupo] = categorias

    return df_resultado, df_resumen, diccionario

In [21]:
# Funcion para eliminar variables altamente correlacionadas
###########################################################

# limpieza por correlación
def _prepare_numeric(df_sub: pd.DataFrame) -> pd.DataFrame:
   return df_sub.apply(pd.to_numeric, errors="coerce")


def _is_constant(series: pd.Series) -> bool:
   # constante si menos de 2 valores no-NaN distintos
   return series.dropna().nunique() < 2


def _safe_corr_with_target(df_num: pd.DataFrame, 
                           feats: list, 
                           target: str, 
                           method: str) -> dict:
   out = {}
   tgt = df_num[target]
   tgt_const = _is_constant(tgt)
   for f in feats:
       s = df_num[f]
       if _is_constant(s) or tgt_const:
           out[f] = 0.0
       else:
           with np.errstate(invalid='ignore', divide='ignore'):
               val = s.corr(tgt, method=method)
           out[f] = 0.0 if (pd.isna(val) or np.isinf(val)) else abs(val)
   return out


def _pair_corr(df_num: pd.DataFrame, 
               a: str, 
               b: str, 
               method: str) -> float:
   sa, sb = df_num[a], df_num[b]
   # si cualquiera es constante, define corr=0 (no dispara eliminación)
   if _is_constant(sa) or _is_constant(sb):
       return 0.0
   pair = pd.concat([sa, sb], axis=1).dropna()
   if pair.shape[0] < 2:
       return 0.0
   with np.errstate(invalid='ignore', divide='ignore'):
       c = pair.corr(method=method).abs().iloc[0, 1]
   if pd.isna(c) or np.isinf(c):
       return 0.0
   return float(c)


def _drops_by_method(df_num: pd.DataFrame, 
                     feats: list, 
                     target: str, 
                     limit: float, 
                     method: str) -> list:
   if len(feats) <= 1 or limit is None:
       return []
   to_remove = set()
   # |corr(feature, target)|
   corr_with_target = _safe_corr_with_target(df_num, feats, target, method)
   n = len(feats)
   corr_dict = {}
   for i in range(n):
       fi = feats[i]
       corr_feature = {}
       if fi in to_remove:
           continue
       for j in range(i + 1, n):
           fj = feats[j]
           if fj in to_remove:
               continue
           corr = _pair_corr(df_num, fi, fj, method)
           corr_feature[fj] = corr 
           if corr >= limit:
               # desempate: conservar mayor |corr con target|
               if corr_with_target.get(fi, 0.0) >= corr_with_target.get(fj, 0.0):
                   to_remove.add(fj)
                   # print(f"Removed {fj} because: {fj}: {corr_with_target.get(fi, 0.0)} >= {corr_with_target.get(fj, 0.0)}")
               else:
                   to_remove.add(fi)
                   # print(f"Removed {fi} because: {fj}: {corr_with_target.get(fi, 0.0)} < {corr_with_target.get(fj, 0.0)}")
                   break
       corr_dict[fi] = corr_feature
   return list(to_remove), corr_dict
    
def select_features_by_correlation(
   df: pd.DataFrame,
   features: list,
   target_column: str,
   pearson_limit: float = 0.9,
   spearman_limit: float = 0.9,
   verbose: bool = True
) -> list:
   """
   Elimina features altamente correlacionadas en dos etapas:
     1) Pearson con umbral `pearson_limit`
     2) Spearman con umbral `spearman_limit`
   Desempate: conserva la que tenga mayor |corr(feature, target)| con el mismo método.
   Manejo extra:
     - Convierte a numérico (errores -> NaN).
     - Ignora columnas constantes o completamente NaN al calcular correlaciones (trata su corr como 0).
   Retorna la lista de features conservadas.
   """
   print("\n\033[1m\033[4mELIMINANDO COLUMNAS NUMERICAS ALTAMENTE CORRELACIONADAS...\033[0m\n")
   cols_needed = list(dict.fromkeys(list(features) + [target_column]))
   df_num = _prepare_numeric(df[cols_needed].copy())

   if verbose:
       const_cols = [c for c in cols_needed if _is_constant(df_num[c])]
       if const_cols:
           print(f"Columnas constantes/degeneradas (tratadas con corr=0): {const_cols}")
   
   # Etapa 1: Pearson
   remaining = sorted(set(features))
   drops_pearson, dict1 = _drops_by_method(df_num, remaining, target_column, pearson_limit, "pearson")
   
   remaining = sorted(set(remaining) - set(drops_pearson))
   # print('Pearson_remaining', remaining)
   print("Columnas eliminadas - correlación Pearson")
   for col in drops_pearson:
       print(f" - {col}")
    
   # Etapa 2: Spearman
   drops_spearman, dict2 = _drops_by_method(df_num, remaining, target_column, spearman_limit, "spearman")
   remaining = sorted(set(remaining) - set(drops_spearman))
   #print('Spearman_remaining', remaining)
   print("\nColumnas eliminadas - correlación Spearman")
   for col in drops_spearman:
       print(f" - {col}")
       
   drops = sorted(set(drops_pearson + drops_spearman))
   print('\nTotal columnas eliminadas: ' + str(len(drops)))

   df = df.drop(columns=drops)
   print(f"\nShape final del DataFrame: {df.shape}")
   print("\nDistribucion de la target:")
   print(df.target_m.value_counts())
   return df

In [22]:
# Funcion para crear nuevas columnas a partir de las columnas de fechas
#######################################################################

def procesar_fechas(df: pd.DataFrame, col_fechas: list) -> tuple:
    """
    Procesa columnas de fechas en un DataFrame:
    - Convierte columnas en col_fechas a datetime.
    - Calcula diferencias en días entre fechas específicas.
    - Devuelve:
        - Un DataFrame modificado con nuevas columnas y sin col_fechas originales.
        - Un listado con las nuevas columnas + fecha_vinculacion + fecha_constitucion.
    """
    print("\n\033[1m\033[4mCREANDO NUEVAS COLUMNAS FECHAS...\033[0m\n")
    
    df = df.copy()  # Evitar modificar el original

    # 1. Convertir columnas en col_fechas a datetime si existen
    for col in col_fechas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    nuevas_columnas = []

    # 2. Diferencia entre fecha_constitucion y fecha_vinculacion
    if 'fecha_vinculacion' in df.columns and 'fecha_constitucion' in df.columns:
        df['dias_entre_constitucion_y_vinculacion'] = (
            df['fecha_vinculacion'] - df['fecha_constitucion']
        ).dt.days
        nuevas_columnas.append('dias_entre_constitucion_y_vinculacion')

    # 3. Diferencia entre fecha mínima de columnas cp_ y fecha_vinculacion
    cp_cols = [col for col in col_fechas if 'cp_' in col and col in df.columns]
    if cp_cols and 'fecha_vinculacion' in df.columns:
        df['cp_fecha_min'] = df[cp_cols].min(axis=1)
        df['dias_entre_cp_min_y_vinculacion'] = (
            df['cp_fecha_min'] - df['fecha_vinculacion']
        ).dt.days
        nuevas_columnas.append('dias_entre_cp_min_y_vinculacion')
        df.drop(columns=['cp_fecha_min'], inplace=True)

    # 4. Diferencia entre fecha mínima de otras columnas y fecha_vinculacion
    otras_cols = [
        col for col in col_fechas
        if col not in cp_cols + ['fecha_constitucion', 'fecha_vinculacion']
        and col in df.columns
    ]
    if otras_cols and 'fecha_vinculacion' in df.columns:
        df['otras_fecha_min'] = df[otras_cols].min(axis=1)
        df['dias_entre_min_fecha_y_vinculacion'] = (
            df['otras_fecha_min'] - df['fecha_vinculacion']
        ).dt.days
        nuevas_columnas.append('dias_entre_min_fecha_y_vinculacion')
        df.drop(columns=['otras_fecha_min'], inplace=True)

    # 5. Eliminar col_fechas del DataFrame
    cols_a_eliminar = [col for col in col_fechas if col in df.columns and col not in ['fecha_constitucion', 'fecha_vinculacion']]
    df.drop(columns=cols_a_eliminar, inplace=True)

    # Calcular estadísticas descriptivas
    df_resumen = df[nuevas_columnas].describe().T

    print('Resumen\n')
    display(df_resumen)

    # agregamos columnas a la lista de columnas fecha
    nuevas_columnas.extend(['fecha_constitucion', 'fecha_vinculacion'])
    
    # 7. Retornar DataFrame modificado y listado de columnas
    return df, nuevas_columnas, df_resumen

# Pre-procesamiento de datos

In [29]:
print("\033[1mPRE-PROCESANDO DF\033[0m\n")

# df
df_1 = df.copy()
# columnas que por decision del negocio deben conservarse para el entrenamiento del modelo
col_conservar = ['num_documento','cp_monto_total_ing', 'cp_promedio_mensual_ing', 'cp_maximo_mensual_ing', 'cp_ultimo_mes_ing', 'cp_monto_total_egr',
                 'cp_promedio_mensual_egr', 'cp_maximo_mensual_egr', 'cp_ultimo_mes_egr', 'cp_ratio_egr', 'cp_monto_pep_ing', 'cp_monto_pep_egr',
                 'cantidad_noticias', 'cp_flag_noticia_ing', 'cp_flag_noticia_egr', 'nivel_riesgo_lsb_total', 'nivel_riesgo_lsb_ultima',
                 'fecha_mas_antiguo_lsb', 'fecha_mas_reciente_lsb', 'entidad_solc_ultimo_lsb', 'cp_fecha_mas_antiguo_lsb_ing',
                 'cp_fecha_mas_reciente_lsb_ing', 'cp_fecha_mas_antiguo_lsb_egr', 'cp_fecha_mas_reciente_lsb_egr', 'cp_cantidad_lsb_egr',
                 'flag_ros_12m', 'flag_ros_hist', 'q_ros_hist', 'cantidad_lsb', 'cp_fecha_ultimo_ros_ing', 'cp_flag_ros_egr','cp_fecha_ultimo_ros_egr',
                 'cp_cantidad_lsb_egr', 'monto_ro_debajo_umbral', 'q_ro_debajo_umbral', 'cp_cant_trx_ing_ros', 'cp_monto_total_ing_ros',
                 'cp_cant_trx_egr_ros', 'cp_monto_total_egr_ros', 'flag_alerta_hist', 'q_alerta_hist', 'cp_flag_desv_egr', 'tipo_alerta_n2','flag_casos_hist'
                 'cantidad_lsb',
 'nivel_riesgo_lsb_total',
 'nivel_riesgo_lsb_ultima','monto_cargos_efectivo_ult_mes'
                ]

# columnas a eliminar
col_eliminar = ['codunico', 'coddocrele', 'key_value', 'target',  'p_codmes']

# columnas de control
col_control = ['target_m', 'mes_base', 'codunico']

# PRE-PROCESADO
df_1 = filtro_columnas_nulas(df_1, 0.2, col_conservar, col_eliminar, 'target_m') # filtramos columnas con alto porcentaje de nulos (conservando col_conservar)
df_1 = generar_target(df_1, 2, 0.1) # generamos la target
col_target, col_fechas, col_categoricas, col_numericas, df_1 = clasificar_columnas(df_1, 'target_m') # clasificamos columnas
#df_1, resumen_imputacion_categoricas = imputar_categoricas(df_1,col_categoricas, 'BAJO', '0','sin_dato') # imputamos variables categoricas
#df_1, resumen_imputacion_numericas = imputar_numericas(df_1, col_numericas, '0', '0', '0', '0') # imputamos variables numericas
#df_1, resumen_cat_reagrupadas, dicc_cat_reagrupadas = reagrupar_categorias(df_1, col_categoricas, 'target_m') # se reagrupan categoricas
df_1 = select_features_by_correlation(df_1, col_numericas, 'target_m', 0.9, 0.9, True) # se eliminan variables altamente correlacionadas
# df, col_fechas, resumen_nuevas_col = procesar_fechas(df, col_fechas) # se crean nuevas columnas con informacion de fechas

# con el df final volvemos a correr la columnas
print("\n\033[1m\033[4mCLASIFICANDO COLUMNAS FINALES...\033[0m\n")
col_target, col_fechas, col_categoricas, col_numericas, df_1 = clasificar_columnas(df_1, 'target_m') # clasificamos columnas

PRE-PROCESANDO DF


FILTRANDO COLUMNAS...

Shape inicial del df: (1433597, 187)

Se eliminan del df las siguientes columnas:
 - codunico
 - p_codmes

Listado de columnas a conservar (no se aplica filtro segun porcentaje de datos faltates):
 - num_documento
 - nivel_riesgo_lsb_total
 - nivel_riesgo_lsb_ultima
 - fecha_mas_antiguo_lsb
 - fecha_mas_reciente_lsb
 - flag_ros_12m
 - flag_ros_hist
 - q_ros_hist
 - cantidad_lsb
 - monto_ro_debajo_umbral
 - q_ro_debajo_umbral
 - cp_cant_trx_ing_ros
 - cp_monto_total_ing_ros
 - cp_cant_trx_egr_ros
 - cp_monto_total_egr_ros
 - flag_alerta_hist
 - q_alerta_hist
 - tipo_alerta_n2
 - nivel_riesgo_lsb_total
 - nivel_riesgo_lsb_ultima
 - target_m

Se eliminan las siguientes columnas por tener valor único:
 - tipo_cliente
 - subsegmento
 - fecha_ejecucion
 - flag_desv_activa
 - flag_cce_r1
 - flag_casos_12m
 - flag_desv_hist
 - flag_alerta_12m

Se eliminan 64 columnas con más del 20.0% de nulos

Shape del df: (1433597, 113)

CREANDO TARGET...

Escenari

In [30]:
df_1.shape

(1433597, 36)